# Plot Figures / Overview analysis

Finalises the atlas labels and metadata

1. **Liver cells.** "Non-adeno" nuclei (from liver biopsies) are sub-clustered and relabelled
   as hepatocytes or cholangiocytes.
2. **Metadata.** Clinical columns are harmonised (smoking group, treatment stage, TP53/EGFR
   spelling) and colours fixed.
3. **Figures.** UMAPs by cell type and clinical variables; per-sample composition with
   clinical annotation tracks; nuclei per sample; CNV heatmaps of malignant cells; agreement
   between the copyKAT and inferCNV malignant calls.


In [ ]:
# !pip install --quiet scanpy infercnvpy seaborn
# %cd <project>/notebooks

In [ ]:
import os

import infercnvpy as cnv
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib.colors import to_hex
from scipy.stats import wilcoxon

## Config

In [ ]:
input_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_inferCNV_add_copyKAT_score.h5ad"

output_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_inferCNV_add_copyKAT_score_ctype_new.h5ad"

figures_output_dir = "./../figures-cellbender/integration/3_annotation/2_clean/Figure_1"
tables_output_dir = "./../results/figure1"
seed = 0

label_col = "cell_type_new"
sample_col = "ID_2"
patient_col = "ID"
counts_layer = "raw_counts"
copykat_col = "copykat_pred"
cnv_score_col = "cnv_score"

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, figsize=(6, 6))

## 1. Load

In [ ]:
adata = sc.read(input_path)
adata.uns.pop("log1p", None)
adata.X = adata.layers[counts_layer].copy()
sc.pp.normalize_total(adata, target_sum=1e4)   # log-normalised X for gene plots; reset to counts before saving
sc.pp.log1p(adata)
adata.obs[label_col].value_counts()

## 2. Relabel liver-biopsy cells

"Non-adeno" nuclei are re-clustered and assigned to hepatocytes or cholangiocytes from liver
markers. The sample table confirms they come from the liver biopsies.

In [ ]:
# --- 1. relabel liver-biopsy cells ("Non-adeno") ---
liver = dict(
    parent="Non-adeno", use_rep="X_scVI_n_latent_10", n_neighbors=30, resolution=0.5,
    mapping={"0": "Hepatocyte", "1": "Cholangiocyte", "2": "Hepatocyte", "3": "Hepatocyte", "4": "Hepatocyte",
             "5": "Hepatocyte", "6": "Cholangiocyte", "7": "Hepatocyte", "8": "Cholangiocyte"},
    expected_n_clusters=9,
)
liver_markers = {
    "Hepatocyte": ["ALB", "AFP", "TTR", "CYP3A4", "ASGR1", "ASGR2", "KRT18"],
    "Cholangiocyte": ["KRT19", "SOX9", "CFTR", "GPBAR1"],
    "LSEC": ["LYVE1", "STAB1", "STAB2"],
    "Kupffer": ["CLEC4F", "MARCO", "CD68"],
}


sub = adata[adata.obs[label_col] == liver["parent"]].copy()
sc.pp.neighbors(sub, n_neighbors=liver["n_neighbors"], use_rep=liver["use_rep"], random_state=seed)
sc.tl.umap(sub, random_state=seed)
sc.tl.leiden(sub, key_added="leiden_sub", resolution=liver["resolution"], flavor="leidenalg", random_state=seed)
n_clusters = sub.obs["leiden_sub"].nunique()
assert n_clusters == liver["expected_n_clusters"], f"{n_clusters} sub-clusters, mapping written for {liver['expected_n_clusters']}"

panels = {k: [g for g in v if g in sub.var_names] for k, v in liver_markers.items()}
sc.pl.dotplot(sub, panels, groupby="leiden_sub", standard_scale="var", save="liver_subclusters_markers.pdf")
sub.obs["new_label"] = sub.obs["leiden_sub"].astype(str).map(liver["mapping"]).astype("category")
sc.pl.umap(sub, color=["leiden_sub", "new_label", sample_col], size=60, wspace=0.4, save="_liver_subclusters.pdf")

labels = adata.obs[label_col].astype(str)
labels.loc[sub.obs_names] = sub.obs["new_label"].astype(str)
adata.obs[label_col] = labels.astype("category")

sc.pl.umap(adata, color=[g for v in liver_markers.values() for g in v if g in adata.var_names] + [patient_col],
           color_map="OrRd", size=20, save="_liverBiopsy.pdf")
pd.crosstab(sub.obs[sample_col], sub.obs["new_label"])

## 3. Harmonise metadata and colours

Value maps fix spelling and group categories. Colours are stored as lists in category order
(scanpy convention) so they survive saving and reloading.

In [ ]:
# --- 2. metadata harmonisation: new_col <- map(source_col); values not in the map are kept ---
metadata_maps = {
    "smoking_status_grp": ("smoking_status", {"ex_smoker": "smoker", "current_smoker": "smoker",
                                              "na": "unknown", "non_smoker ": "non_smoker"}),
    "recist_v2": ("recist_v1", {"adj_normal": "0_adj_normal", "treatment_naive": "1_treatment_naive",
                                "treatment_naive_d": "2_treatment_naive", "partial_response": "3_on_treatment",
                                "stable_disease": "3_on_treatment", "TKI_resistance": "4_TKI_resistance"}),
    "recist_v3": ("recist_v1", {"adj_normal": "0_adj_normal", "treatment_naive": "1_treatment_naive",
                                "treatment_naive_d": "1_treatment_naive", "partial_response": "3_on_treatment",
                                "stable_disease": "3_on_treatment", "TKI_resistance": "4_TKI_resistance"}),
    "p53_mutation": ("p53_mutation", {"na": "unknown", "wt ": "wt", "misense": "missense"}),
    "EGFR_mutation": ("EGFR_mutation", {"Exon_19_del": "exon19del"}),
    copykat_col: (copykat_col, {"nan": "not.defined", "unknown": "not.defined"}),   # not run / filtered -> not.defined
}

# --- 3. colours ---
cell_type_colors = {
    "Malignant": "#023fa5", "Non-malignant epithelial": "#84bcf0", "Transitional": "#63ffac",
    "Alveolar cell type I": "#7d87b9", "Alveolar cell type II": "#bec1d4", "Club cell": "#4a6fe3",
    "Goblet cell": "#8e063b", "Ciliated epithelial": "#bb7784", "Hepatocyte": "#d6bcc0",
    "Cholangiocyte": "#0291a5", "Endothelial": "#93019f", "Fibroblast": "#e07b91",
    "Myeloid": "#d33f6a", "Lymphoid": "#ef9708",
}
recist_colors = {"0_adj_normal": "#d86464", "1_treatment_naive": "#f4c9c9",
                 "3_on_treatment": "#af8b8b", "4_TKI_resistance": "#636262"}
copykat_colors = {"diploid": "#FFE4C4", "aneuploid": "#023fa5", "not.defined": "#afafac"}



for new_col, (src_col, mapping) in metadata_maps.items():
    adata.obs[new_col] = adata.obs[src_col].astype(str).replace(mapping).astype("category")
    print(f"{new_col}: {dict(adata.obs[new_col].value_counts())}")


def set_colors(col, palette):
    cats = list(adata.obs[col].cat.categories)
    missing = [c for c in cats if c not in palette]
    assert not missing, f"{col}: no colour for {missing}"
    adata.uns[f"{col}_colors"] = [palette[c] for c in cats]


set_colors(label_col, cell_type_colors)
set_colors(stage_col, recist_colors)
adata.obs[copykat_col] = adata.obs[copykat_col].cat.reorder_categories(
    [c for c in copykat_colors if c in adata.obs[copykat_col].cat.categories])
set_colors(copykat_col, copykat_colors)

## 4. UMAPs (Figures)

In [ ]:
# --- 4. figures ---
umap_panels = ["ID", "stage_at_diagnosis", "p53_mutation", "EGFR_mutation", "smoking_status_grp", "recist_v3", copykat_col]
stage_col = "recist_v3"
# sample order on the x-axis of per-sample plots (adjacent normal, naive, on treatment, other, resistant by TKI generation)
sample_order = [
    "TN008_AN", "TN004_AN", "TN006_AN", "TN007_AN", "TN001_AN", "TN002_AN", "TN005_AN", "TN003_AN", "TN009_AN",
    "TN008", "TN004", "TN006", "TN007", "TN001", "TN002", "TN005", "TN003", "TN009",
    "P003", "P002", "P006", "P008", "P012", "P005", "P013", "P001", "P009", "P011",
    "O001", "E001", "D003", "D002", "D001",
    "R006", "R003", "R001", "R010",     # 1st-generation TKI
    "R009",                             # 2nd-generation TKI
    "R007", "R005", "R008", "R012",     # 3rd-generation TKI
    "R004", "R002", "R011",
]

sc.pl.umap(adata, color=label_col, size=20, legend_loc="on data", legend_fontsize="x-small",
           legend_fontoutline=True, save="_annotated_major_cell_type.pdf")
for col in umap_panels:
    sc.pl.umap(adata, color=col, size=20, legend_fontsize="x-small", legend_fontoutline=True, save=f"_{col}.pdf")
for stage in adata.obs[stage_col].cat.categories:
    sc.pl.umap(adata, color=stage_col, groups=[stage], size=25, save=f"_{stage_col}_{stage}.pdf")

## 5. Composition per sample

Stacked cell-type proportions per sample (Malignant at the bottom), with clinical annotation
tracks above the bars, and the number of nuclei per sample.

In [ ]:
order = [s for s in dict.fromkeys(sample_order) if s in set(adata.obs[sample_col])]
missing = sorted(set(adata.obs[sample_col]) - set(order))
if missing:
    print(f"samples not in sample_order, appended: {missing}")
order += missing

props = pd.crosstab(adata.obs[sample_col], adata.obs[label_col], normalize="index").mul(100).loc[order]
props = props[["Malignant"] + [c for c in props.columns if c != "Malignant"]]
props.round(2).to_csv(f"{tables_output_dir}/{label_col}_percent_per_{sample_col}.csv")


annotation_tracks = {"EGFR mutation": ("EGFR_mutation", "Dark2"), "TP53 mutation": ("p53_mutation", "Set3"),
                     "Smoking": ("smoking_status_grp", "tab10")}   # track title: (obs column, colormap)
heatmap_vmin, heatmap_vmax = -0.2, 0.5
min_cells_per_group = 20                # per-sample copyKAT vs inferCNV comparison


tracks = {}
for title, (col, cmap) in annotation_tracks.items():
    pairs = adata.obs[[sample_col, col]].astype(str).drop_duplicates()
    multi = pairs[pairs[sample_col].duplicated(keep=False)]
    assert multi.empty, f"{col}: several values within a sample: {multi.values.tolist()}"
    values = pairs.set_index(sample_col)[col]
    cats = sorted(values.unique())
    palette = dict(zip(cats, [to_hex(c) for c in plt.get_cmap(cmap).colors]))
    tracks[title] = (values, palette)

fig, ax = plt.subplots(figsize=(12, 5))
props.plot.bar(stacked=True, ax=ax, width=0.85, edgecolor="none", legend=False,
               color=[cell_type_colors[c] for c in props.columns])
box_h, gap, extra = 4, 0.5, []
for i, (title, (values, palette)) in enumerate(tracks.items()):
    y = 101 + i * (box_h + gap)
    for j, s in enumerate(props.index):
        extra.append(ax.add_patch(mpatches.Rectangle((j - 0.4, y), 0.8, box_h, color=palette[values[s]], clip_on=False)))
    extra.append(ax.text(len(props) - 0.3, y + box_h / 2, title, va="center", fontsize=8))
ax.set_ylim(0, 100)
ax.set_ylabel(f"{label_col} (%)")
ax.set_xlabel("")
legends = [("Cell type", {c: cell_type_colors[c] for c in props.columns})] +           [(title, palette) for title, (_, palette) in tracks.items()]
y = 1.0
for title, palette in legends:                # stacked legends to the right, in axes coordinates
    leg = plt.matplotlib.legend.Legend(ax, [mpatches.Patch(color=v) for v in palette.values()], list(palette),
                                       title=title, loc="upper left", bbox_to_anchor=(1.12, y),
                                       frameon=False, fontsize=7, title_fontsize=8)
    ax.add_artist(leg)
    extra.append(leg)
    y -= 0.055 * (len(palette) + 2)
plt.savefig(f"{figures_output_dir}/barplot_{sample_col}_proportions_{label_col}_annotated.pdf",
            bbox_inches="tight", bbox_extra_artists=extra, dpi=300)
plt.show()

n_nuclei = adata.obs[sample_col].value_counts().loc[order]
n_nuclei.to_csv(f"{tables_output_dir}/nuclei_per_{sample_col}.csv")
fig, ax = plt.subplots(figsize=(12, 2.5))
n_nuclei.plot.bar(ax=ax, color="black", width=0.85)
ax.set_ylabel("Nuclei")
sns.despine(ax=ax)
plt.savefig(f"{figures_output_dir}/barplot_num_nuclei_{sample_col}.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 6. CNV heatmaps

Malignant cells by treatment stage and by stage at diagnosis, and all cells by final label.

In [ ]:
malignant = adata[adata.obs[label_col] == "Malignant"]
for col in ["recist_v2", "stage_at_diagnosis"]:
    cnv.pl.chromosome_heatmap(malignant, groupby=col, use_rep="cnv", vmin=heatmap_vmin, vmax=heatmap_vmax,
                              figsize=(11, 5), save=f"_malignant_{col}.pdf")
cnv.pl.chromosome_heatmap_summary(malignant, groupby="recist_v2", use_rep="cnv", vmax=0.1, figsize=(11, 2.2),
                                  save="_malignant_recist_v2_summary.pdf")
cnv.pl.chromosome_heatmap(adata, groupby=label_col, use_rep="cnv", vmin=heatmap_vmin, vmax=heatmap_vmax,
                          figsize=(11, 10), save=f"_{label_col}_all.pdf")

## 7. copyKAT vs inferCNV

- CNV score of copyKAT-aneuploid vs diploid cells

In [ ]:
obs = adata.obs
fig, ax = plt.subplots(figsize=(3.5, 1.8))
sns.boxplot(data=obs, y=copykat_col, x=cnv_score_col, hue=copykat_col, palette=copykat_colors,
            order=list(copykat_colors), legend=False, fliersize=0, ax=ax)
ax.set_xlabel("inferCNV score")
ax.set_ylabel("copyKAT")
plt.savefig(f"{figures_output_dir}/copyKAT_vs_inferCNV_score.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
sns.countplot(data=obs, x="cnv_status_cluster", hue=copykat_col, palette=copykat_colors,
              hue_order=list(copykat_colors), ax=ax)
ax.set_ylabel("Nuclei")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/bycnvCluster_copyKAT_vs_inferCNV.pdf", bbox_inches="tight")
plt.show()

label_order = [c for c in obs[label_col].cat.categories if c != "Malignant"] + ["Malignant"]
counts = pd.crosstab(obs[label_col], obs[copykat_col]).loc[label_order]
counts.to_csv(f"{tables_output_dir}/copykat_counts_per_{label_col}.csv")
split = 20000
fig, (top, bottom) = plt.subplots(2, 1, sharex=True, figsize=(6, 7), height_ratios=[1, 2.5])
for ax in (top, bottom):
    sns.countplot(data=obs, x=label_col, order=label_order, hue=copykat_col, hue_order=list(copykat_colors),
                  palette=copykat_colors, ax=ax, legend=ax is top)
top.set_ylim(split, counts.values.max() * 1.05)
bottom.set_ylim(0, split * 0.75)
top.spines["bottom"].set_visible(False)
bottom.spines["top"].set_visible(False)
top.tick_params(bottom=False)
kw = dict(marker=[(-1, -0.1), (1, 0.1)], markersize=5, linestyle="none", color="k", clip_on=False)
top.plot([0, 1], [0, 0], transform=top.transAxes, **kw)
bottom.plot([0, 1], [1, 1], transform=bottom.transAxes, **kw)
top.set_ylabel("")
bottom.set_ylabel("Nuclei")
bottom.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/bycellType_copyKAT_vs_inferCNV_broken.pdf", bbox_inches="tight")
plt.show()
counts

## 8. Combined label and save (X = raw counts)

In [ ]:
adata.obs["cell_type_copykat"] = (adata.obs[label_col].astype(str) + "_" + adata.obs[copykat_col].astype(str)).astype("category")
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
adata.write(output_path)
adata.obs[label_col].value_counts()